In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import FortranFile
import smplotlib
from hmf import MassFunction
from astropy.cosmology import FlatLambdaCDM
import pandas as pd

findfont: Failed to find font weight normal, now using 500.


# Read the Luminosity Bin File

In [3]:
inputs = "/Users/dexbhadra/Documents/delphi/input_z4p5Tree_files/"

In [4]:
with FortranFile(inputs + "lum_all_cueto_kroupa_all_Z_delphi.bin", "r") as f:
    # 4 header records, one number each
    n_wave = int(f.read_ints(np.int64)[0])   # 7 channels
    n_step = int(f.read_ints(np.int64)[0])   # 30 age steps of 30 Myr
    n_z    = int(f.read_ints(np.int64)[0])   # 44 redshifts (the tree snapshots)
    n_Z    = int(f.read_ints(np.int64)[0])   # 100 metallicities
    # then one record of 100 numbers (all metallicities) for every channel, age step and redshift,
    # in the same loop order delphi.f90 uses to read it (lines 206-212)
    lum = np.zeros((n_wave, n_step, n_z, n_Z))
    for iwave in range(n_wave):
        for istep in range(n_step):
            for iz in range(n_z):
                lum[iwave, istep, iz, :] = f.read_reals(np.float64)
print(n_wave, n_step, n_z, n_Z)
print(lum.shape)

7 30 44 100
(7, 30, 44, 100)


In [5]:
# the 44 redshifts: from the redshift grid of the merger trees
with FortranFile(inputs + "redshift_grid_mh8to14.bin", "r") as f:
    _ = f.read_ints(np.int32)
    redshifts = f.read_reals(np.float64)

# the 100 metallicities: from the header of the IMF table (delphi.f90 lines 157-158 do the same)
with FortranFile(inputs + "table_cueto_kroupa_imf_m2.bin", "r") as f:
    _ = f.read_ints(np.int64)                 # 100
    log_Z_lowest = f.read_reals(np.float64)[0] # -6.0
    log_Z_step = f.read_reals(np.float64)[0]   # 0.0434
log_Z = log_Z_lowest + np.arange(n_Z) * log_Z_step

print(redshifts[:5], redshifts[-1])   # 4.5, 4.58, ...  40.76
print(log_Z[:3], log_Z[-1])           # -6.0, -5.957, ...  -1.7

[4.5     4.58308 4.66934 4.75899 4.85224] 40.76271
[-6.         -5.95656566 -5.91313131] -1.6999999999999833


In [6]:
L1500 = lum[2, 0]          # channel 2 = 1500 A, age step 0 = 0-30 Myr old; shape (44 redshifts, 100 metallicities)
iZ = 76                   
for iz in range(44):
    print(f"z = {redshifts[iz]:.2f}   Z = {10**log_Z[iZ]:.5f}   L1500 = {L1500[iz, iZ]:.4e}")

z = 4.50   Z = 0.00200   L1500 = 1.5395e+40
z = 4.58   Z = 0.00200   L1500 = 1.5549e+40
z = 4.67   Z = 0.00200   L1500 = 1.5701e+40
z = 4.76   Z = 0.00200   L1500 = 1.5850e+40
z = 4.85   Z = 0.00200   L1500 = 1.5997e+40
z = 4.95   Z = 0.00200   L1500 = 1.6142e+40
z = 5.05   Z = 0.00200   L1500 = 1.6286e+40
z = 5.16   Z = 0.00200   L1500 = 1.6427e+40
z = 5.27   Z = 0.00200   L1500 = 1.6568e+40
z = 5.38   Z = 0.00200   L1500 = 1.6707e+40
z = 5.50   Z = 0.00200   L1500 = 1.6844e+40
z = 5.63   Z = 0.00200   L1500 = 1.6981e+40
z = 5.76   Z = 0.00200   L1500 = 1.7194e+40
z = 5.90   Z = 0.00200   L1500 = 1.7511e+40
z = 6.05   Z = 0.00200   L1500 = 1.7826e+40
z = 6.20   Z = 0.00200   L1500 = 1.8139e+40
z = 6.37   Z = 0.00200   L1500 = 1.8451e+40
z = 6.54   Z = 0.00200   L1500 = 1.8762e+40
z = 6.73   Z = 0.00200   L1500 = 1.9072e+40
z = 6.92   Z = 0.00200   L1500 = 1.9383e+40
z = 7.13   Z = 0.00200   L1500 = 1.9693e+40
z = 7.35   Z = 0.00200   L1500 = 2.0003e+40
z = 7.59   Z = 0.00200   L1500 =

The IMF starts rising or turns top-heavy once the L1500 values change. eSFE and fiducial models need a IMF that is Krouppa and not a function of redshift and metallicity. So we need to find the metallicity at z=4.5 where f_massive <=0 in order to be able to replicate the Krouppa IMF.

In [7]:
def f_massive(log_Z, z):            # paper Eq. 8
    x = 1 + log_Z - np.log10(0.0134)
    return 1.07 * (1 - 2**x) + 0.04 * 2.67**x * z

kroupa = f_massive(log_Z[None, :], redshifts[:, None]) <= 0     # shape (44 redshifts, 100 metallicities), True = Kroupa
print("Kroupa cells:", kroupa.sum(), "of", kroupa.size)
print("Kroupa metallicities at z = 4.5:", kroupa[0].sum(), "-> Z from", 10**log_Z[kroupa[0]][0], "to", 10**log_Z[kroupa[0]][-1])

Kroupa cells: 488 of 4400
Kroupa metallicities at z = 4.5: 21 -> Z from 0.002699685234971679 to 0.019952623149689562


In [8]:
kroupa_rows = f_massive(log_Z, redshifts[0]) <= 0     # True for the 21 metallicities where the IMF is Kroupa at z = 4.5
Z_kroupa = 10**log_Z[kroupa_rows]                     # their metallicities (as Z, not log Z): 0.0027 ... 0.02
n_Z_kroupa = len(Z_kroupa)                            # 21
print(n_Z_kroupa, Z_kroupa, Z_kroupa)

21 [0.00269969 0.00298365 0.00329748 0.00364432 0.00402764 0.00445128
 0.00491948 0.00543693 0.0060088  0.00664083 0.00733933 0.00811131
 0.00896448 0.0099074  0.01094949 0.0121012  0.01337404 0.01478077
 0.01633546 0.01805368 0.01995262] [0.00269969 0.00298365 0.00329748 0.00364432 0.00402764 0.00445128
 0.00491948 0.00543693 0.0060088  0.00664083 0.00733933 0.00811131
 0.00896448 0.0099074  0.01094949 0.0121012  0.01337404 0.01478077
 0.01633546 0.01805368 0.01995262]


In [27]:
wwith FortranFile(inputs + "lum_kroupa_from_cueto_table.bin", "w") as f:
    # records 1-4: the header. delphi.f90 reads these as 4-byte integers, hence np.int32
    f.write_record(np.int32(n_wave))       # 7 channels                       (line 181)
    f.write_record(np.int32(n_step))       # 30 age steps                     (line 183, must be 30)
    f.write_record(np.int32(n_z))          # 44 redshifts                     (line 189, must match the trees)
    f.write_record(np.int32(n_Z_kroupa))   # 21 metallicities                 (line 215)
    # record 5: the metallicities                                             (line 217)
    f.write_record(Z_kroupa)
    # then, for each metallicity and each channel, one record of the 30 age steps at z = 4.5   (lines 219-223)
    for iZ in np.where(kroupa_rows)[0]:     # the indices of the 21 Kroupa rows: 79, 80, ..., 99
        for iwave in range(n_wave):
            f.write_record(lum[iwave, :, 0, iZ])